In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from tqdm.auto import tqdm

from GG4 import Brain
from task.wk_3.BMI_and_Hand import BMI_and_Hand

In [ ]:
# Replace with your control policy.
# observations : list[np.ndarray] – neural measurements y_0 … y_{t-1}
# target       : tuple[float, float] – desired hand position (x, y) in cm
# current_pos  : tuple[float, float] – current hand position (x, y) in cm
# returns      : array-like – control input u for this step

def control_policy(observations, target, current_pos):
    raise NotImplementedError

In [ ]:
T           = 1000
T_WARMUP    = 100
N_TRIALS    = 8
TARGET_X    = 0.0
TARGET_Y    = 50.0
DEMO_SEED   = 0
N_SNAPSHOTS = 6
ARM_LINK    = 30.0
MAX_DELTA   = 0.25

In [ ]:
def _ik(x, y, prev_sh=0.0, prev_el=0.0):
    r2 = x**2 + y**2
    cos_el = np.clip((r2 - 2 * ARM_LINK**2) / (2 * ARM_LINK**2), -1.0, 1.0)
    el_pos = float(np.arccos(cos_el))
    el_neg = -el_pos

    def _sh(el):
        s = np.arctan2(y, x) - np.arctan2(
            ARM_LINK * np.sin(el), ARM_LINK + ARM_LINK * np.cos(el)
        )
        return float(np.arctan2(np.sin(s), np.cos(s)))

    def _wrap_dist(a, b):
        d = a - b
        return float(np.arctan2(np.sin(d), np.cos(d)))

    sh_pos, sh_neg = _sh(el_pos), _sh(el_neg)
    err_pos = _wrap_dist(sh_pos, prev_sh)**2 + _wrap_dist(el_pos, prev_el)**2
    err_neg = _wrap_dist(sh_neg, prev_sh)**2 + _wrap_dist(el_neg, prev_el)**2
    return (sh_pos, el_pos) if err_pos <= err_neg else (sh_neg, el_neg)

In [ ]:
def _run_trial(seed):
    bmi = BMI_and_Hand(Brain(random_seed=seed))
    target = (TARGET_X, TARGET_Y)
    target_xy = np.array([TARGET_X, TARGET_Y])
    observations = []
    hand_traj = np.zeros((T, 2))
    sh_traj   = np.zeros(T)
    el_traj   = np.zeros(T)
    prev_sh = prev_el = 0.0

    for t in range(T):
        current_pos = bmi.hand_pos
        y = np.array(bmi._brain.measure())
        u = control_policy(observations, target, current_pos)
        observations.append(y)
        bmi.next_state(np.asarray(u, dtype=float).tolist())
        x_h, y_h = bmi.hand_pos
        hand_traj[t] = [x_h, y_h]
        sh, el = _ik(x_h, y_h, prev_sh, prev_el)
        sh_traj[t] = sh if abs(sh - prev_sh) <= MAX_DELTA else prev_sh
        el_traj[t] = el if abs(el - prev_el) <= MAX_DELTA else prev_el
        prev_sh, prev_el = sh_traj[t], el_traj[t]

    dist = np.linalg.norm(hand_traj - target_xy, axis=1)
    return hand_traj, sh_traj, el_traj, dist


all_hand = []
all_sh   = []
all_el   = []
all_dist = np.zeros((N_TRIALS, T))

seeds = [DEMO_SEED] + [s for s in range(N_TRIALS) if s != DEMO_SEED]
seeds = seeds[:N_TRIALS]

with tqdm(total=N_TRIALS, desc="Evaluating", unit="trial") as pbar:
    for i, seed in enumerate(seeds):
        ht, sh, el, dist = _run_trial(seed)
        all_hand.append(ht)
        all_sh.append(sh)
        all_el.append(el)
        all_dist[i] = dist
        pbar.set_postfix(seed=seed, final_err=f"{dist[-1]:.1f} cm")
        pbar.update(1)

demo_hand = all_hand[0]
demo_sh   = all_sh[0]
demo_el   = all_el[0]
demo_dist = all_dist[0]

print(f"Final distance  mean \u00b1 std: "
      f"{all_dist[:, -1].mean():.2f} \u00b1 {all_dist[:, -1].std():.2f} cm")

In [ ]:
t_axis    = np.arange(T)
dist_mean = all_dist.mean(axis=0)
dist_std  = all_dist.std(axis=0)

fig, ax = plt.subplots()

for d in all_dist:
    ax.plot(t_axis, d, color="steelblue", alpha=0.20, lw=0.8)

ax.plot(t_axis, dist_mean, color="steelblue", lw=2.0, label="Mean")
ax.fill_between(
    t_axis,
    dist_mean - dist_std,
    dist_mean + dist_std,
    color="steelblue", alpha=0.20, label=r"$\pm$1 s.d.",
)
ax.axvline(T_WARMUP, color="grey", ls="--", lw=0.9, label="Warm-up")

ax.set_xlabel("Time (steps)")
ax.set_ylabel("Distance to target (cm)")
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
post   = slice(T_WARMUP, T)
traj   = demo_hand[post]
sh_p   = demo_sh[post]
el_p   = demo_el[post]
n_post = traj.shape[0]
t_norm = np.linspace(0.0, 1.0, n_post)

cmap   = plt.get_cmap("plasma")
norm_c = Normalize(vmin=0, vmax=1)

fig, ax = plt.subplots()

theta_ws = np.linspace(0, 2 * np.pi, 300)
ax.fill(
    2 * ARM_LINK * np.cos(theta_ws),
    2 * ARM_LINK * np.sin(theta_ws),
    color="grey", alpha=0.07, zorder=0, label="Reachable workspace",
)

sc = ax.scatter(
    traj[:, 0], traj[:, 1],
    c=t_norm, cmap="plasma", s=6, zorder=3, linewidths=0,
)
cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Time (normalised)")

snap_idx = np.round(np.linspace(0, n_post - 1, N_SNAPSHOTS)).astype(int)
for k, idx in enumerate(snap_idx):
    sh_k, el_k = sh_p[idx], el_p[idx]
    ex = ARM_LINK * np.cos(sh_k)
    ey = ARM_LINK * np.sin(sh_k)
    hx = ex + ARM_LINK * np.cos(sh_k + el_k)
    hy = ey + ARM_LINK * np.sin(sh_k + el_k)
    frac    = k / max(N_SNAPSHOTS - 1, 1)
    alpha_k = 0.25 + 0.55 * frac
    color_k = cmap(norm_c(frac))
    ax.plot([0, ex, hx], [0, ey, hy],
            color=color_k, lw=1.5, alpha=alpha_k, zorder=2)
    ax.scatter([0, ex, hx], [0, ey, hy],
               color=color_k, s=20, alpha=alpha_k, zorder=2)

ax.scatter(0, 0, s=90, color="k", zorder=6)

ax.scatter(
    *traj[0], s=160, marker="o",
    facecolors="white", edgecolors="C0", lw=2, zorder=5,
    label="Start",
)
ax.scatter(
    *traj[-1], s=160, marker="s",
    facecolors="white", edgecolors="C2", lw=2, zorder=5,
    label="End",
)
ax.scatter(
    TARGET_X, TARGET_Y,
    s=320, marker="*", color="red", edgecolors="darkred",
    lw=0.5, zorder=5, label="Target",
)

final_err = float(demo_dist[-1])
err_circle = plt.Circle(
    (TARGET_X, TARGET_Y), final_err,
    color="red", fill=False, ls="--", lw=0.8, alpha=0.45, zorder=1,
)
ax.add_patch(err_circle)

ax.set_aspect("equal")
ax.set_xlabel("$x$ (cm)")
ax.set_ylabel("$y$ (cm)")
ax.legend(loc="upper right", framealpha=0.9)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()